# Pandas Workbook: from first principles to interview-ready

**How to use this**
1. Read the short idea at the top of each section, then run the code cells **one at a time**.
2. Wherever you see **🔮 Predict**, write your guess on paper *before* running the cell. Being wrong is where the learning happens.
3. Wherever you see **🧩 Try first**, solve it yourself, then run the next cell to compare with the reference.
4. After Part 13, open `pandas_exam.ipynb`, answer it, and run the checker. Then send me your answers and I'll explain every mistake.

**Map**
| Part | Topic |
|---|---|
| 0 | Mental model + pandas 3 changes |
| 1 | Series, DataFrame, dtypes |
| 2 | Selection: `[]`, `loc`, `iloc`, masks, assignment traps |
| 3 | Index alignment (the thing that surprises everyone) |
| 4 | Missing data |
| 5 | **Data profiling** |
| 6 | **Data cleaning** |
| 7 | GroupBy: agg / transform / apply / filter |
| 8 | Merge, join, concat |
| 9 | Reshaping: pivot, melt, stack, explode |
| 10 | Time series |
| 11 | Vectorization, `map`/`apply`/`replace`, `cut`/`qcut`, ranking |
| 12 | Performance + memory |
| 13 | Interview classics (with reasoning) |

## Part 0: The mental model

Everything in pandas follows from three ideas:

1. **A `Series` is a NumPy array plus a label for every position** (the *index*).
2. **A `DataFrame` is a dict of Series that share one index.** Each column has its own dtype.
3. **Operations align on labels, not positions.** This is the biggest difference from NumPy and the source of most "why is my result full of NaN?" bugs.

If you remember only one sentence: *pandas asks "which label?" before it asks "which position?"*

> **pandas 3.0 note** (you have 3.0.x installed): Copy-on-Write is now always on, and text columns use a dedicated `str` dtype by default instead of `object`. Many older tutorials and interview answers still describe the old behaviour, so I flag differences as we go.

In [1]:
import pandas as pd
import numpy as np
import warnings
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
print(pd.__version__)

3.0.2


## Part 1: Series, DataFrame, dtypes

Why dtypes matter: they decide memory, speed, and which operations are legal. A column of numbers stored as strings is the #1 cleaning problem you will ever meet.

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="score")
print(s, "\n")
print("values :", s.values)
print("index  :", s.index)
print("dtype  :", s.dtype)

df = pd.DataFrame({
    "name":   ["Asha", "Ravi", "Meera", "Kiran"],
    "age":    [25, 32, 41, 29],
    "salary": [50.5, 60.0, 75.25, 58.0],
    "joined": pd.to_datetime(["2020-01-15", "2019-06-01", "2018-03-20", "2021-09-10"]),
})
df.info()

a    10
b    20
c    30
Name: score, dtype: int64 

values : [10 20 30]
index  : Index(['a', 'b', 'c'], dtype='str')
dtype  : int64
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   name    4 non-null      str           
 1   age     4 non-null      int64         
 2   salary  4 non-null      float64       
 3   joined  4 non-null      datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 260.0 bytes


**🔮 Predict:** what are the dtypes of these three Series? Then run.

In [3]:
print(pd.Series([1, 2, 3]).dtype)
print(pd.Series([1, 2, None]).dtype)          # why float?
print(pd.Series([1, 2, None], dtype="Int64").tolist())   # nullable integer
print(pd.Series([True, False, None]).dtype)   # careful
print(pd.Series(["a", "b"]).dtype)            # 'str' in pandas 3, 'object' in pandas 2

int64
float64
[1, 2, <NA>]
object
str


**Why?** NumPy integers cannot hold "missing", so one `None` forces the whole column to `float64` (NaN is a float). The nullable dtypes (`Int64`, `boolean`, `string`) use `pd.NA` and keep integers as integers.

**Interview line:** *"Why did my integer column become float after a merge?"* → The merge introduced missing values, and a plain NumPy integer column cannot hold NaN.

## Part 2: Selection (the part interviewers love)

| Tool | Selects by | Slice end | Use for |
|---|---|---|---|
| `df[...]` | column name(s), boolean mask, or row **slice** | n/a | quick column grabs |
| `.loc` | **labels** | **inclusive** | almost everything |
| `.iloc` | **positions** | exclusive (like Python) | when you truly mean "the 3rd row" |
| `.at` / `.iat` | single label / position | n/a | fast scalar access |

In [4]:
emp = pd.DataFrame({
    "name":   ["Asha", "Ravi", "Meera", "Kiran", "Neha"],
    "dept":   ["Eng", "Eng", "HR", "Sales", "HR"],
    "salary": [90, 120, 65, 60, 50],
}, index=[10, 20, 30, 40, 50])
emp

,name,dept,salary
10,Asha,Eng,90
20,Ravi,Eng,120
30,Meera,HR,65
40,Kiran,Sales,60
50,Neha,HR,50


**🔮 Predict** how many rows each returns:

In [5]:
print(len(emp.loc[10:30]))     # labels 10..30 inclusive
print(len(emp.iloc[0:3]))      # positions 0,1,2
print(len(emp[1:3]))           # df[slice] is POSITIONAL
# emp[10] would raise KeyError (it looks for a column named 10)

3
3
2


### Trap 1: integer index + `s[1]`
With an integer index, `s[1]` means *label* 1, which may not be the second element. Always say `.loc` or `.iloc`.

In [6]:
t = pd.Series([5, 6, 7, 8], index=[3, 2, 1, 0])
print("loc[1]  =", t.loc[1])    # label 1
print("iloc[1] =", t.iloc[1])   # second element

loc[1]  = 7
iloc[1] = 6


### Trap 2: boolean masks, `&` / `|`, and parentheses
Python's `and`/`or` do not work on Series. Use `&`, `|`, `~`, and **parenthesise each condition** because `&` binds tighter than `>`.

In [7]:
mask = (emp.salary > 55) & (emp.dept != "HR")
print(emp[mask])
# emp[emp.salary > 55 & emp.dept != "HR"]   # TypeError / wrong precedence
print(emp.query("salary > 55 and dept != 'HR'"))   # query() allows plain `and`
print(emp[emp.dept.isin(["HR", "Sales"])])
print(emp[emp.salary.between(60, 100)])            # inclusive on both ends by default

     name   dept  salary
10   Asha    Eng      90
20   Ravi    Eng     120
40  Kiran  Sales      60
     name   dept  salary
10   Asha    Eng      90
20   Ravi    Eng     120
40  Kiran  Sales      60
     name   dept  salary
30  Meera     HR      65
40  Kiran  Sales      60
50   Neha     HR      50
     name   dept  salary
10   Asha    Eng      90
30  Meera     HR      65
40  Kiran  Sales      60


### Trap 3: assigning to a slice (SettingWithCopy → Copy-on-Write)
**Old behaviour (pandas ≤2):** `df[mask]["col"] = x` sometimes worked, sometimes warned.
**pandas 3:** it *never* modifies the original. Always assign with a single `.loc[mask, col]`.

In [8]:
a = emp.copy()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a[a.salary > 55]["bonus"] = 1          # chained: operates on a temporary copy
print("chained assignment worked? ->", "bonus" in a.columns)

a.loc[a.salary > 55, "bonus"] = 1          # correct
print(a)

chained assignment worked? -> False
     name   dept  salary  bonus
10   Asha    Eng      90    1.0
20   Ravi    Eng     120    1.0
30  Meera     HR      65    1.0
40  Kiran  Sales      60    1.0
50   Neha     HR      50    NaN


### Trap 4: "view vs copy" in the interview
Under Copy-on-Write, every derived object behaves like a copy (pandas shares memory lazily and copies only when you modify). So the modern answer is: *"You can't accidentally modify the parent through a child, so `.copy()` is rarely needed for safety."*

In [9]:
b = emp["salary"]            # looks like a view
b.iloc[0] = 999              # modifies only b under CoW
print(emp.loc[10, "salary"]) # still 90

90


## Part 3: Index alignment

Arithmetic between pandas objects matches labels first. Labels present in only one side give NaN.

In [10]:
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30], index=["b", "c", "d"])
print(s1 + s2)

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64


**🔮 Predict** `(s1 + s2).sum()` before running the next cell.

In [11]:
print((s1 + s2).sum())                 # NaN skipped by sum
print(s1.add(s2, fill_value=0))        # treat missing as 0 instead

35.0
a     1.0
b    12.0
c    23.0
d    30.0
dtype: float64


### The classic DataFrame version
Two frames with the same *values* but different row order:

In [12]:
x = pd.DataFrame({"v": [1, 2, 3]}, index=[0, 1, 2])
y = pd.DataFrame({"v": [10, 20, 30]}, index=[2, 1, 0])
print(x + y)                    # aligned by label: 1+30, 2+20, 3+10
print(x.values + y.values)      # positional: 1+10, 2+20, 3+30

    v
0  31
1  22
2  13
[[11]
 [22]
 [33]]


**Takeaway:** when you assign a Series as a new column (`df["new"] = other_series`) it is also aligned on the index. If you want positional assignment use `.values` or `reset_index(drop=True)` on both.

## Part 4: Missing data

- `np.nan` (float), `None` (object), `pd.NaT` (datetime) and `pd.NA` (nullable) all count as missing.
- `NaN != NaN`, so `x == np.nan` is always False. Use `.isna()`.

In [13]:
x = pd.Series([1, np.nan, 3])
print((x == np.nan).tolist())
print(x.isna().tolist())

print(pd.Series([np.nan, np.nan]).sum())              # 0.0, not NaN!
print(pd.Series([np.nan, np.nan]).sum(min_count=1))   # NaN
print(x.mean())          # skipna=True -> (1+3)/2 = 2.0, NOT 4/3
print(x.mean(skipna=False))

[False, False, False]
[False, True, False]
0.0
nan
2.0
nan


### GroupBy silently drops NaN keys

In [14]:
g = pd.DataFrame({"k": ["A", "A", None, "B", None], "v": [1, 2, 3, 4, 5]})
print(g.groupby("k")["v"].sum())                  # 3 + 4, rows with NaN key vanish
print(g.groupby("k", dropna=False)["v"].sum())    # keep them

k
A    3
B    4
Name: v, dtype: int64
k
A      3
B      4
NaN    8
Name: v, dtype: int64


### Filling strategies and when each is defensible
| Method | Good for | Danger |
|---|---|---|
| constant (`0`, `"Unknown"`) | counts, categories | 0 is a real value for numbers |
| mean / median | roughly symmetric / skewed numeric | shrinks variance, ignores groups |
| **group median** via `transform` | numeric that varies by group | needs enough rows per group |
| `ffill` / `bfill` | ordered data (time series) | must sort first; do it *within* a group |
| interpolate | smooth numeric time series | invents data |

In [15]:
d = pd.DataFrame({
    "team":  ["A", "A", "A", "B", "B", "B"],
    "score": [10, np.nan, 20, np.nan, np.nan, 30],
})
d["global_mean_fill"] = d.score.fillna(d.score.mean())
d["group_mean_fill"]  = d.score.fillna(d.groupby("team")["score"].transform("mean"))
d

,team,score,global_mean_fill,group_mean_fill
0,A,10.0,10.0,10.0
1,A,NaN,20.0,15.0
2,A,20.0,20.0,20.0
3,B,NaN,20.0,30.0
4,B,NaN,20.0,30.0
5,B,30.0,30.0,30.0


## Part 5: Data profiling

**Profiling = understanding a dataset before touching it.** A good profile answers:

1. **Structure:** rows, columns, dtypes, memory.
2. **Completeness:** missing counts and percentages. Also *disguised* missing values (`"N/A"`, `"-"`, `0`, `999`, `""`).
3. **Uniqueness:** duplicate rows, duplicate keys, cardinality of each column.
4. **Validity:** values outside allowed range (negative quantity, age 200), wrong formats.
5. **Consistency:** the same thing spelled several ways (`"Delhi"`, `"delhi "`, `"DELHI"`).
6. **Distribution:** skew, outliers.

Here is a deliberately dirty dataset that we will profile and then clean:

In [16]:
raw = pd.DataFrame({
    "order_id":   [1001, 1002, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009],
    "customer":   ["  Asha ", "ravi", "ravi", "MEERA", "Meera", None, "Kiran", "asha", "Ravi ", "Kiran"],
    "city":       ["Delhi", "delhi", "delhi", "Mumbai", "mumbai ", "Pune", "PUNE", "Delhi", "Mumbai", np.nan],
    "amount":     ["1,200", "850", "850", "₹2,000", "-50", "3000", "N/A", "999999", "700", "450"],
    "order_date": ["2024-01-05", "05/01/2024", "05/01/2024", "2024-02-10", "2024-02-30",
                   "2024-03-15", "2024-03-18", "2024-04-01", "2024-04-05", "2024-04-09"],
    "qty":        [2, 1, 1, 5, 1, 3, 2, 1, 200, -1],
    "rating":     [4.5, 3.0, 3.0, np.nan, 5.0, 4.0, np.nan, 2.5, 3.5, 4.0],
})
raw

,order_id,customer,city,amount,order_date,qty,rating
0,1001,Asha,Delhi,"1,200",2024-01-05,2,4.5
1,1002,ravi,delhi,850,05/01/2024,1,3.0
2,1002,ravi,delhi,850,05/01/2024,1,3.0
3,1003,MEERA,Mumbai,"₹2,000",2024-02-10,5,NaN
4,1004,Meera,mumbai,-50,2024-02-30,1,5.0
5,1005,NaN,Pune,3000,2024-03-15,3,4.0
6,1006,Kiran,PUNE,N/A,2024-03-18,2,NaN
7,1007,asha,Delhi,999999,2024-04-01,1,2.5
8,1008,Ravi,Mumbai,700,2024-04-05,200,3.5
9,1009,Kiran,NaN,450,2024-04-09,-1,4.0


### 5.1 First look

In [17]:
print(raw.shape)
raw.info()
print(raw.memory_usage(deep=True).sum(), "bytes")

(10, 7)
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    10 non-null     int64  
 1   customer    9 non-null      str    
 2   city        9 non-null      str    
 3   amount      10 non-null     str    
 4   order_date  10 non-null     str    
 5   qty         10 non-null     int64  
 6   rating      8 non-null      float64
dtypes: float64(1), int64(2), str(4)
memory usage: 692.0 bytes
2551 bytes


**🧩 Try first:** `amount` should be numeric but looks like `str`. What does `raw.describe()` show for it? (Hint: describe only picks numeric columns by default.)

In [18]:
print(raw.describe())                 # amount is missing from this table: a dtype red flag
print(raw.describe(include="all").T)  # everything, transposed to read easily

          order_id         qty    rating
count    10.000000   10.000000  8.000000
mean   1004.700000   21.500000  3.687500
std       2.750757   62.737991  0.842509
min    1001.000000   -1.000000  2.500000
25%    1002.250000    1.000000  3.000000
50%    1004.500000    1.500000  3.750000
75%    1006.750000    2.750000  4.125000
max    1009.000000  200.000000  5.000000
           count unique         top freq    mean        std     min      25%     50%      75%     max
order_id    10.0    NaN         NaN  NaN  1004.7   2.750757  1001.0  1002.25  1004.5  1006.75  1009.0
customer       9      7        ravi    2     NaN        NaN     NaN      NaN     NaN      NaN     NaN
city           9      6       Delhi    2     NaN        NaN     NaN      NaN     NaN      NaN     NaN
amount        10      9         850    2     NaN        NaN     NaN      NaN     NaN      NaN     NaN
order_date    10      9  05/01/2024    2     NaN        NaN     NaN      NaN     NaN      NaN     NaN
qty         10.0   

### 5.2 A reusable profiling function

In [19]:
def profile(df):
    """One row per column: type, completeness, uniqueness, and a sample value."""
    return pd.DataFrame({
        "dtype":      df.dtypes.astype(str),
        "non_null":   df.notna().sum(),
        "missing_%":  (df.isna().mean() * 100).round(1),
        "nunique":    df.nunique(),
        "sample":     df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

profile(raw)

,dtype,non_null,missing_%,nunique,sample
order_id,int64,10,0.0,9,1001
customer,str,9,10.0,7,Asha
city,str,9,10.0,6,Delhi
amount,str,10,0.0,9,"1,200"
order_date,str,10,0.0,9,2024-01-05
qty,int64,10,0.0,6,2
rating,float64,8,20.0,6,4.5


### 5.3 Duplicates: full-row vs key-based

In [20]:
print("full-row duplicates      :", raw.duplicated().sum())
print("duplicate order_id       :", raw.duplicated("order_id").sum())
print(raw[raw.duplicated("order_id", keep=False)])   # keep=False shows ALL copies

full-row duplicates      : 1
duplicate order_id       : 1
   order_id customer   city amount  order_date  qty  rating
1      1002     ravi  delhi    850  05/01/2024    1     3.0
2      1002     ravi  delhi    850  05/01/2024    1     3.0


**Interview trap:** `duplicated()` marks the 2nd, 3rd... copy by default (`keep="first"`). If you want to *see* every copy use `keep=False`.

### 5.4 Inconsistent categories (look *after* normalising)

In [21]:
print(raw["city"].value_counts(dropna=False))
print("--- after strip + lower ---")
print(raw["city"].str.strip().str.lower().value_counts(dropna=False))

city
Delhi      2
delhi      2
Mumbai     2
mumbai     1
Pune       1
PUNE       1
NaN        1
Name: count, dtype: int64
--- after strip + lower ---
city
delhi     4
mumbai    3
pune      2
NaN       1
Name: count, dtype: int64


### 5.5 Validity checks (business rules)

In [22]:
amount_num = pd.to_numeric(raw["amount"].str.replace(r"[₹,]", "", regex=True), errors="coerce")
checks = {
    "qty <= 0":               (raw.qty <= 0).sum(),
    "amount <= 0":            (amount_num <= 0).sum(),
    "amount unparseable":     amount_num.isna().sum(),
    "rating outside 1..5":    (~raw.rating.dropna().between(1, 5)).sum(),
}
print(pd.Series(checks))

qty <= 0               1
amount <= 0            1
amount unparseable     1
rating outside 1..5    0
dtype: int64


### 5.6 Outliers: the IQR rule
Flag values beyond `Q1 − 1.5·IQR` or `Q3 + 1.5·IQR`. Median-based and robust, unlike mean ± 3·std (which is itself distorted by the outlier you are hunting).

In [23]:
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

valid_amount = amount_num.where(amount_num > 0)
print(raw.assign(amount_num=amount_num)[iqr_flags(valid_amount)][["order_id", "amount_num"]])
print("mean:", valid_amount.mean().round(1), "| median:", valid_amount.median())

   order_id  amount_num
7      1007    999999.0
mean: 126131.1 | median: 1025.0


**Reasoning check:** the mean is dragged above 100k by a single value while the median stays near 1k. This is why profiling reports *both*.

## Part 6: Data cleaning

Principles (say these in an interview):
1. **Never edit the raw data.** Build a function `raw → clean` so it is repeatable.
2. **Standardise first, then deduplicate.** `"Ravi "` and `"ravi"` only collapse after normalising.
3. **Convert, don't assume.** `errors="coerce"` turns bad values into NaN, which you then *count and inspect*, not silently ignore.
4. **Flag, don't always delete.** Keep an `is_outlier` column; decide later.
5. **Log what you changed** (row counts before/after).

### 6.1 Strings

In [24]:
s = pd.Series(["  Asha ", "RAVI", None, "mEEra kumar"])
print(s.str.strip().str.title().tolist())
print(s.str.contains("a", case=False, na=False).tolist())   # na=False avoids NaN in a mask
print(s.str.extract(r"(\w+)\s*$").iloc[:, 0].tolist())      # last word via regex group

['Asha', 'Ravi', nan, 'Meera Kumar']
[True, True, False, True]
['Asha', 'RAVI', nan, 'kumar']


### 6.2 Numbers stored as text

In [25]:
amt = pd.Series(["1,200", "₹2,000", "N/A", "850"])
print(pd.to_numeric(amt.str.replace(r"[₹,]", "", regex=True), errors="coerce").tolist())
# pd.to_numeric(amt) without cleaning would raise ValueError
# Indian lakh format "1,00,000" also works with the same replace, because we drop ALL commas.

[1200.0, 2000.0, nan, 850.0]


### 6.3 Dates: mixed formats and impossible dates
Safest pattern: parse each known format *explicitly* and combine, so day/month ambiguity is never guessed.

In [26]:
dates = pd.Series(["2024-01-05", "05/01/2024", "2024-02-30", "garbage"])
iso   = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
dmy   = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
print(iso.combine_first(dmy).tolist())      # Feb 30 and 'garbage' become NaT

[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, NaT]


**Interview trap:** `"05/01/2024"` is 5 January in India/UK but 1 May in the US. Never let pandas guess; state the format.

### 6.4 The whole pipeline in one function

In [27]:
def clean_orders(df):
    out = (
        df
        .assign(
            customer   = lambda d: d.customer.str.strip().str.title(),
            city       = lambda d: d.city.str.strip().str.title(),
            amount     = lambda d: pd.to_numeric(d.amount.str.replace(r"[₹,]", "", regex=True), errors="coerce"),
            order_date = lambda d: pd.to_datetime(d.order_date, format="%Y-%m-%d", errors="coerce")
                                      .combine_first(pd.to_datetime(d.order_date, format="%d/%m/%Y", errors="coerce")),
        )
        .drop_duplicates()
    )
    # business rules -> NaN (keep the row, drop the bad value)
    out["amount"] = out["amount"].where(out["amount"] > 0)
    out["qty"]    = out["qty"].where(out["qty"] > 0)
    # flags
    out["amount_outlier"] = iqr_flags(out["amount"])
    # fill: city with a label, rating with the customer's own median, then overall median
    out["city"]   = out["city"].fillna("Unknown")
    out["rating"] = (out["rating"]
                     .fillna(out.groupby("customer")["rating"].transform("median"))
                     .fillna(out["rating"].median()))
    return out.reset_index(drop=True)

clean = clean_orders(raw)
print(f"rows: {len(raw)} -> {len(clean)}")
clean

rows: 10 -> 9


,order_id,customer,city,amount,order_date,qty,rating,amount_outlier
0,1001,Asha,Delhi,1200.0,2024-01-05,2.0,4.5,False
1,1002,Ravi,Delhi,850.0,2024-01-05,1.0,3.0,False
2,1003,Meera,Mumbai,2000.0,2024-02-10,5.0,5.0,False
3,1004,Meera,Mumbai,NaN,NaT,1.0,5.0,False
4,1005,NaN,Pune,3000.0,2024-03-15,3.0,4.0,False
5,1006,Kiran,Pune,NaN,2024-03-18,2.0,4.0,False
6,1007,Asha,Delhi,999999.0,2024-04-01,1.0,2.5,True
7,1008,Ravi,Mumbai,700.0,2024-04-05,200.0,3.5,False
8,1009,Kiran,Unknown,450.0,2024-04-09,NaN,4.0,False


In [28]:
profile(clean)       # run the same profile again: the before/after comparison is your evidence

,dtype,non_null,missing_%,nunique,sample
order_id,int64,9,0.0,9,1001
customer,str,8,11.1,4,Asha
city,str,9,0.0,4,Delhi
amount,float64,7,22.2,7,1200.0
order_date,datetime64[us],8,11.1,7,2024-01-05 00:00:00
qty,float64,8,11.1,5,2.0
rating,float64,9,0.0,6,4.5
amount_outlier,bool,9,0.0,2,False


**🧩 Think:** the `customer` column still has one `NaN`. Is dropping that row right? (Answer: depends on the business. For revenue totals keep it as `"Unknown"`; for customer-level modelling drop it. Always state the assumption.)

### 6.5 Handy cleaning one-liners

In [29]:
df = pd.DataFrame({"A ": [1, 2], "B-c": [3, 4], "A": [5, 6]})
df.columns = df.columns.str.strip().str.lower().str.replace(r"\W+", "_", regex=True)
print(df.columns.tolist())                  # note: 'a' now appears twice
print(df.loc[:, ~df.columns.duplicated()])  # keep first of each duplicate column name

p = pd.Series(["a", "b", "c", "a"])
print(p.replace({"a": "X"}).tolist())               # replace: unmatched stay unchanged
print(p.map({"a": "X"}).tolist())                   # map: unmatched become NaN   <-- classic gotcha

['a', 'b_c', 'a']
   a  b_c
0  1    3
1  2    4
['X', 'b', 'c', 'X']
['X', nan, nan, 'X']


## Part 7: GroupBy (split → apply → combine)

Think of it as three stages: **split** rows into groups, **apply** a function to each group, **combine** results.
What changes between `agg`, `transform`, `filter`, `apply` is *what shape comes back*:

| Method | Returns | Shape |
|---|---|---|
| `agg` | one row per group | smaller |
| `transform` | one value per original row | **same as input** |
| `filter` | the original rows of groups that pass a test | subset |
| `apply` | anything (flexible, slowest) | varies |

In [30]:
e = pd.DataFrame({
    "emp":    ["Asha", "Ravi", "Meera", "Kiran", "Neha", "Arjun", "Divya", "Sana", "Vikram", "Isha", "Rohan", "Tara"],
    "dept":   ["Eng", "Eng", "Eng", "Eng", "HR", "HR", "HR", "Sales", "Sales", "Sales", "Sales", "Ops"],
    "salary": [90, 120, 120, 70, 50, 65, 65, 60, 80, 80, 55, 48],
})
print(e.groupby("dept")["salary"].agg(["count", "mean", "max"]))

# Named aggregation: the readable modern form
print(e.groupby("dept").agg(headcount=("emp", "count"), avg_sal=("salary", "mean"), top=("salary", "max")))

       count    mean  max
dept                     
Eng        4  100.00  120
HR         3   60.00   65
Ops        1   48.00   48
Sales      4   68.75   80


       headcount  avg_sal  top
dept                          
Eng            4   100.00  120
HR             3    60.00   65
Ops            1    48.00   48
Sales          4    68.75   80


### transform: attach a group statistic to every row

In [31]:
e["dept_avg"]  = e.groupby("dept")["salary"].transform("mean")
e["pct_of_dept"] = (e.salary / e.groupby("dept")["salary"].transform("sum") * 100).round(1)
e["vs_avg"]    = e.salary - e["dept_avg"]
e.head(6)

,emp,dept,salary,dept_avg,pct_of_dept,vs_avg
0,Asha,Eng,90,100.0,22.5,-10.0
1,Ravi,Eng,120,100.0,30.0,20.0
2,Meera,Eng,120,100.0,30.0,20.0
3,Kiran,Eng,70,100.0,17.5,-30.0
4,Neha,HR,50,60.0,27.8,-10.0
5,Arjun,HR,65,60.0,36.1,5.0


### filter: keep whole groups that satisfy a condition

In [32]:
print(e.groupby("dept").filter(lambda g: g["salary"].mean() > 65)["dept"].unique())
print(e.groupby("dept").filter(lambda g: len(g) >= 3)["dept"].unique())

<StringArray>
['Eng', 'Sales']
Length: 2, dtype: str
<StringArray>
['Eng', 'HR', 'Sales']
Length: 3, dtype: str


### Interview classic A: top-N per group
Sort, then `head` within group. Faster and clearer than `apply`.

In [33]:
top2 = e.sort_values("salary", ascending=False).groupby("dept").head(2).sort_values(["dept", "salary"], ascending=[True, False])
print(top2[["dept", "emp", "salary"]])

     dept     emp  salary
1     Eng    Ravi     120
2     Eng   Meera     120
5      HR   Arjun      65
6      HR   Divya      65
11    Ops    Tara      48
9   Sales    Isha      80
8   Sales  Vikram      80


### Interview classic B: Nth highest *distinct* salary per department (ties!)
`rank(method="dense")` collapses ties: 120,120,90 → ranks 1,1,2. `"min"` gives 1,1,3, `"first"` gives 1,2,3.

In [34]:
e["r_dense"] = e.groupby("dept")["salary"].rank(method="dense", ascending=False)
e["r_min"]   = e.groupby("dept")["salary"].rank(method="min",   ascending=False)
e["r_first"] = e.groupby("dept")["salary"].rank(method="first",  ascending=False)
print(e[e.dept == "Eng"][["emp", "salary", "r_dense", "r_min", "r_first"]])
print("2nd highest distinct per dept:")
print(e[e.r_dense == 2][["dept", "emp", "salary"]])     # Ops has none: only one distinct salary

     emp  salary  r_dense  r_min  r_first
0   Asha      90      2.0    3.0      3.0
1   Ravi     120      1.0    1.0      1.0
2  Meera     120      1.0    1.0      2.0
3  Kiran      70      3.0    4.0      4.0
2nd highest distinct per dept:
    dept   emp  salary
0    Eng  Asha      90
4     HR  Neha      50
7  Sales  Sana      60


### Interview classic C: average of averages ≠ overall average

In [35]:
t = pd.DataFrame({"store": ["X", "X", "X", "Y"], "sales": [10, 10, 10, 100]})
print("mean of group means:", t.groupby("store")["sales"].mean().mean())   # (10 + 100) / 2
print("true overall mean  :", t["sales"].mean())                           # 130 / 4
# Weighted mean with explicit weights:
w = pd.DataFrame({"g": ["a", "a"], "price": [10, 20], "units": [90, 10]})
print(np.average(w.price, weights=w.units))

mean of group means: 55.0
true overall mean  : 32.5
11.0


### Interview classic D: groupby + apply pitfalls
`apply` can be slow and its output shape depends on what the function returns. Prefer `agg`/`transform` with built-ins. Use `apply` only when you need a function of *several columns at once*.

In [36]:
def weighted_price(g):
    return np.average(g["price"], weights=g["units"])

shop = pd.DataFrame({"cat": ["a", "a", "b", "b"], "price": [10, 20, 5, 15], "units": [90, 10, 50, 50]})
print(shop.groupby("cat")[["price", "units"]].apply(weighted_price))

cat
a    11.0
b    10.0
dtype: float64


## Part 8: Merge, join, concat

`merge` = SQL join. `join` = merge on the index. `concat` = stack frames (rows or columns) aligned on the other axis.

In [37]:
orders = pd.DataFrame({"order_id": range(1, 9), "cust_id": [1, 1, 2, 3, 3, 3, 5, 9],
                       "amount": [100, 250, 80, 300, 120, 60, 500, 40]})
customers = pd.DataFrame({"cust_id": [1, 2, 3, 4, 6], "cname": ["Asha", "Ravi", "Meera", "Kiran", "Neha"]})

for how in ["inner", "left", "right", "outer"]:
    print(how, "->", len(orders.merge(customers, on="cust_id", how=how)))

inner -> 6
left -> 8
right -> 8
outer -> 10


### The `indicator` flag and anti-joins (a favourite question)

In [38]:
m = orders.merge(customers, on="cust_id", how="outer", indicator=True)
print(m["_merge"].value_counts())
print("orders without a customer :", m.loc[m._merge == "left_only", "order_id"].tolist())
print("customers without orders  :", m.loc[m._merge == "right_only", "cname"].tolist())

# Alternative anti-join using isin:
print(customers[~customers.cust_id.isin(orders.cust_id)].cname.tolist())

_merge
both          6
left_only     2
right_only    2
Name: count, dtype: int64
orders without a customer : [7.0, 8.0]
customers without orders  : ['Kiran', 'Neha']
['Kiran', 'Neha']


### Row explosion: the silent bug
If the key is duplicated on **both** sides, you get the Cartesian product of those rows.

In [39]:
a = pd.DataFrame({"k": [1, 1], "x": ["a1", "a2"]})
b = pd.DataFrame({"k": [1, 1, 1], "y": ["b1", "b2", "b3"]})
print(len(a.merge(b, on="k")))                 # 2 x 3 = 6
try:
    a.merge(b, on="k", validate="one_to_one")  # defend against it
except Exception as err:
    print(type(err).__name__, "->", err)

6
MergeError -> Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
  k
 1 ...
Duplicates in right:
  k
 1
 1 ...


**Habit to build:** after every merge, check `len()` before vs after, and use `validate="one_to_one"` / `"many_to_one"` when you know the relationship.

### Other merge details

In [40]:
left  = pd.DataFrame({"id": [1, 2], "v": ["l1", "l2"]})
right = pd.DataFrame({"key": [1, 2], "v": ["r1", "r2"]})
print(left.merge(right, left_on="id", right_on="key", suffixes=("_left", "_right")))

# concat aligns on the other axis, filling gaps with NaN
p = pd.DataFrame({"a": [1], "b": [2]})
q = pd.DataFrame({"b": [3], "c": [4]})
print(pd.concat([p, q], ignore_index=True))

   id v_left  key v_right
0   1     l1    1      r1
1   2     l2    2      r2
     a  b    c
0  1.0  2  NaN
1  NaN  3  4.0


## Part 9: Reshaping

- **Wide → long:** `melt`. **Long → wide:** `pivot` / `pivot_table`.
- `pivot` fails on duplicate (index, column) pairs. `pivot_table` aggregates them.

In [41]:
sales = pd.DataFrame({
    "region":  ["N", "N", "N", "S", "S", "S", "E", "E"],
    "product": ["A", "B", "A", "A", "C", "C", "B", "A"],
    "units":   [5, 3, 2, 7, 4, 6, 8, 1],
})
try:
    sales.pivot(index="region", columns="product", values="units")
except ValueError as err:
    print("pivot ->", err)

pt = sales.pivot_table(index="region", columns="product", values="units", aggfunc="sum", fill_value=0, margins=True)
pt

pivot ->

 Index contains duplicate entries, cannot reshape


product,A,B,C,All
region,,,,
E,1,8,0,9
N,7,3,0,10
S,7,0,10,17
All,15,11,10,36


In [42]:
wide = pd.DataFrame({"id": [1, 2], "math": [90, 80], "sci": [70, 60]})
long = wide.melt(id_vars="id", var_name="subject", value_name="score")
print(long)
print(long.pivot(index="id", columns="subject", values="score"))   # round trip

   id subject  score
0   1    math     90
1   2    math     80
2   1     sci     70
3   2     sci     60
subject  math  sci
id                
1          90   70
2          80   60


### `explode`, `crosstab`, `stack`/`unstack`

In [43]:
tags = pd.DataFrame({"id": [1, 2, 3], "tags": ["a,b", "b", "c,a,b"]})
print(tags.assign(tag=tags.tags.str.split(",")).explode("tag")["tag"].value_counts().to_dict())

print(pd.crosstab(sales.region, sales.product))                    # counts of rows
print(pd.crosstab(sales.region, sales.product, normalize="index").round(2))  # row %

multi = sales.groupby(["region", "product"])["units"].sum()        # Series with a MultiIndex
print(multi.unstack(fill_value=0))                                  # inner index level -> columns

{'b': 3, 'a': 2, 'c': 1}
col_0   <bound method DataFrame.prod of   region product  units\n0      N       A      5\n1      N       B      3\n2      N       A      2\n3      S       A      7\n4      S       C      4\n5      S       C      6\n6      E       B      8\n7      E       A      1>
region                                                                                                                                                                                                                                                                  
E                                                       2                                                                                                                                                                                                               
N                                                       3                                                                                                                           

col_0   <bound method DataFrame.prod of   region product  units\n0      N       A      5\n1      N       B      3\n2      N       A      2\n3      S       A      7\n4      S       C      4\n5      S       C      6\n6      E       B      8\n7      E       A      1>
region                                                                                                                                                                                                                                                                  
E                                                     1.0                                                                                                                                                                                                               
N                                                     1.0                                                                                                                                                    

## Part 10: Time series

In [44]:
ts = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=14, freq="D"),
    "rev":  [10, 12, 9, 15, 20, 18, 25, 30, 28, 22, 19, 35, 40, 38],
}).set_index("date")
print(ts.index.min().day_name())                 # 2024-01-01 is a Monday

print(ts.resample("W").sum())                    # 'W' = weeks ENDING Sunday
print(ts.resample("MS").sum())                   # month start

Monday
            rev
date           
2024-01-07  109
2024-01-14  212
            rev
date           
2024-01-01  321


**🔮 Predict:** how many NaN does `rolling(3).mean()` produce, and why?

In [45]:
r = ts["rev"].rolling(3).mean()
print(r.isna().sum())                        # first window-1 rows lack a full window
print(ts["rev"].rolling(3, min_periods=1).mean().head(3).tolist())

ts["prev"]   = ts["rev"].shift(1)            # yesterday
ts["change"] = ts["rev"].diff()              # today - yesterday
ts["pct"]    = ts["rev"].pct_change().round(3)
ts["cum"]    = ts["rev"].cumsum()
ts.head(4)

2
[10.0, 11.0, 10.333333333333334]


,rev,prev,change,pct,cum
date,,,,,
2024-01-01,10,NaN,NaN,NaN,10
2024-01-02,12,10.0,2.0,0.200,22
2024-01-03,9,12.0,-3.0,-0.250,31
2024-01-04,15,9.0,6.0,0.667,46


### Interview classic E: longest consecutive-login streak
Trick: after sorting, a *new streak* starts whenever the gap to the previous date isn't 1 day. A cumulative sum of those "new streak" flags gives each streak an ID.

In [46]:
logins = pd.DataFrame({
    "user": ["u1"] * 5 + ["u2"] * 4 + ["u3"] * 4,
    "date": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-05", "2024-01-06",
             "2024-01-01", "2024-01-03", "2024-01-05", "2024-01-07",
             "2024-01-10", "2024-01-11", "2024-01-11", "2024-01-12"],
})
l = (logins.assign(date=pd.to_datetime(logins.date))
           .drop_duplicates()                       # u3 logged in twice on the 11th
           .sort_values(["user", "date"]))
new_streak = l.groupby("user")["date"].diff().dt.days.ne(1)    # NaT-diff -> NaN -> ne(1) True: first row starts a streak
l["streak_id"] = new_streak.groupby(l["user"]).cumsum()
print(l.groupby(["user", "streak_id"]).size().groupby("user").max())

user
u1    3
u2    1
u3    3
dtype: int64


## Part 11: Vectorisation, `map` / `apply` / `replace`, binning

**Rule:** if a vectorised/built-in method exists, use it. `apply(axis=1)` is a Python loop in disguise.

In [47]:
import time
big = pd.DataFrame({"a": np.random.rand(200_000), "b": np.random.rand(200_000)})

t0 = time.perf_counter(); r1 = big.apply(lambda r: r.a + r.b, axis=1); t1 = time.perf_counter()
r2 = big.a + big.b;                                                      t2 = time.perf_counter()
print(f"apply(axis=1): {t1-t0:.2f}s   vectorised: {t2-t1:.4f}s")

apply(axis=1): 1.85s   vectorised: 0.0007s


### Conditional columns

In [48]:
sc = pd.DataFrame({"marks": [35, 52, 68, 91]})
sc["pass"]  = np.where(sc.marks >= 40, "P", "F")                           # 2 outcomes
sc["grade"] = np.select([sc.marks >= 85, sc.marks >= 60, sc.marks >= 40],   # first True wins
                        ["A", "B", "C"], default="F")
print(sc)
print(sc.marks.where(sc.marks > 50))        # keep where True, else NaN
print(sc.marks.mask(sc.marks > 50, 0))      # replace where True

   marks pass grade
0     35    F     F
1     52    P     C
2     68    P     B
3     91    P     A
0     NaN
1    52.0
2    68.0
3    91.0
Name: marks, dtype: float64
0    35
1     0
2     0
3     0
Name: marks, dtype: int64


### `cut` vs `qcut`, and the bin-edge trap
- `cut`: you choose the edges. **Intervals are right-inclusive by default**: `(0, 17]`.
- `qcut`: pandas chooses edges so each bin has ~equal *counts*.

In [49]:
age = pd.Series([5, 17, 18, 35, 64, 65, 90])
print(pd.cut(age, bins=[0, 17, 64, 200], labels=["child", "adult", "senior"]).tolist())
print(pd.cut(age, bins=[0, 17, 64, 200], right=False, labels=["child", "adult", "senior"]).tolist())  # same edges, [a,b) intervals: 17 and 64 move up a bin
print(pd.qcut(pd.Series(range(1, 9)), q=4, labels=["Q1", "Q2", "Q3", "Q4"]).value_counts().sort_index().tolist())

['child', 'child', 'adult', 'adult', 'adult', 'senior', 'senior']
['child', 'adult', 'adult', 'adult', 'senior', 'senior', 'senior']
[2, 2, 2, 2]


### Other things asked often

In [50]:
v = pd.Series(["x", "y", "x", "z", "x"])
print(v.value_counts(normalize=True).round(2).to_dict())
print(v.nunique(), v.unique().tolist())
sal = pd.Series([50, 90, 90, 70], index=["a", "b", "c", "d"])
print(sal.idxmax())                         # label of FIRST max
print(sal.nlargest(2).index.tolist())       # ties: keep='first'
print(sal.sort_values(ascending=False, kind="stable").index.tolist())

# agg with multiple functions, and DataFrame.map (elementwise on a whole frame)
print(pd.DataFrame({"a": [1, 2], "b": [3, 4]}).map(lambda x: x * 10))

{'x': 0.6, 'y': 0.2, 'z': 0.2}
3 ['x', 'y', 'z']
b
['b', 'c']
['b', 'c', 'd', 'a']
    a   b
0  10  30
1  20  40


## Part 12: Performance and memory

In [51]:
n = 300_000
d = pd.DataFrame({
    "city": np.random.choice(["Delhi", "Mumbai", "Pune", "Chennai"], n),
    "qty":  np.random.randint(0, 100, n),
})
before = d.memory_usage(deep=True).sum()
d["city"] = d["city"].astype("category")           # repeated strings -> small integer codes
d["qty"]  = pd.to_numeric(d["qty"], downcast="unsigned")
after = d.memory_usage(deep=True).sum()
print(f"{before/1e6:.1f} MB -> {after/1e6:.1f} MB")

18.8 MB -> 0.6 MB


**Checklist for big data:** read only needed columns (`usecols`), specify `dtype`, parse dates at read time (`parse_dates`), process in `chunksize`, use `category` for low-cardinality text, avoid `apply(axis=1)`, and avoid building a DataFrame by appending rows in a loop (collect in a list, build once).

In [52]:
# read_csv options interviewers expect you to know
import io
csv = io.StringIO("id,name,joined,score\n1,Asha,2024-01-05,N/A\n2,Ravi,2024-02-10,88\n")
x = pd.read_csv(csv, parse_dates=["joined"], na_values=["N/A", "-"], dtype={"id": "int32"}, usecols=["id", "joined", "score"])
print(x.dtypes)

id                 int32
joined    datetime64[us]
score            float64
dtype: object


## Part 13: Interview classics (reasoning-first)

Cover the answers and say them aloud.

| Question | Strong answer |
|---|---|
| `loc` vs `iloc`? | `loc` = labels, end-inclusive; `iloc` = positions, end-exclusive. |
| `merge` vs `join` vs `concat`? | `merge`: SQL-style on columns. `join`: merge on index. `concat`: stack along an axis, aligning on the other axis. |
| `apply` vs `map` vs `applymap`? | `Series.map`: elementwise (dict/func). `Series/DataFrame.apply`: along an axis. Elementwise on a DataFrame is `DataFrame.map` (`applymap` is deprecated). |
| `agg` vs `transform`? | `agg` reduces to one row per group; `transform` keeps original length. |
| `pivot` vs `pivot_table`? | `pivot` reshapes only, errors on duplicates; `pivot_table` aggregates. |
| How do you handle missing data? | Profile first; decide by column meaning; flag; fill by group; never fill blindly. |
| Why is `x == np.nan` False? | NaN ≠ NaN by IEEE 754. Use `isna()`. |
| Why did my int column turn float? | Missing values introduced; use nullable `Int64`. |
| `sort_values` vs `sort_index`? | by column values vs by labels. |
| `copy()` needed? | Under pandas 3 Copy-on-Write you rarely need it for safety. |
| `inplace=True` good? | Mostly myth: usually still copies; breaks method chaining; discouraged. |
| `drop_duplicates` which row is kept? | `keep="first"` default; `last` or `False` available. |
| Fastest way to loop? | Don't. Vectorise; else `itertuples` beats `iterrows`; `apply(axis=1)` is a loop. |

### Coding puzzles: try each *before* running the solution

**Puzzle 1.** Keep only the **latest** row per user.

In [53]:
ev = pd.DataFrame({"user": list("aabbbc"), "status": list("xypqrz"), "ts": [1, 3, 2, 5, 4, 9]})
# YOUR ATTEMPT HERE

In [54]:
print(ev.sort_values("ts").drop_duplicates("user", keep="last").sort_values("user"))
# Alternative: ev.loc[ev.groupby("user")["ts"].idxmax()]

  user status  ts
1    a      y   3
3    b      q   5
5    c      z   9


**Puzzle 2.** Running total of `amount` per customer, in `order_id` order, as a new column.

In [55]:
o = orders.sort_values("order_id").copy()
o["running"] = o.groupby("cust_id")["amount"].cumsum()
print(o)

   order_id  cust_id  amount  running
0         1        1     100      100
1         2        1     250      350
2         3        2      80       80
3         4        3     300      300
4         5        3     120      420
5         6        3      60      480
6         7        5     500      500
7         8        9      40       40


**Puzzle 3.** Fill missing `score` with the **team's** mean (you saw it in Part 4). Then, for a team with *all* values missing, fall back to the overall mean.

In [56]:
d2 = pd.DataFrame({"team": list("AAABBB"), "score": [10, np.nan, 20, np.nan, np.nan, np.nan]})
d2["score"] = d2.score.fillna(d2.groupby("team")["score"].transform("mean")).fillna(d2.score.mean())
print(d2)

  team  score
0    A   10.0
1    A   15.0
2    A   20.0
3    B   15.0
4    B   15.0
5    B   15.0


**Puzzle 4.** Number of **distinct** customers per day, and the cumulative number of distinct customers seen so far.

In [57]:
v = pd.DataFrame({"day": [1, 1, 2, 2, 3], "cust": ["a", "b", "a", "c", "c"]})
daily = v.groupby("day")["cust"].nunique()
first_seen = v.drop_duplicates("cust").groupby("day").size()     # new customers per day
print(daily.tolist(), first_seen.reindex(daily.index, fill_value=0).cumsum().tolist())

[2, 2, 1] [2, 3, 3]


**Puzzle 5.** Employees earning more than their department average.

In [58]:
e2 = e[["emp", "dept", "salary"]]
print(e2[e2.salary > e2.groupby("dept")["salary"].transform("mean")])

      emp   dept  salary
1    Ravi    Eng     120
2   Meera    Eng     120
5   Arjun     HR      65
6   Divya     HR      65
8  Vikram  Sales      80
9    Isha  Sales      80


**Puzzle 6.** Without a loop, find rows where `value` is higher than **both** neighbours.

In [59]:
w = pd.Series([1, 5, 2, 8, 3, 3, 9, 4])
peaks = (w > w.shift(1)) & (w > w.shift(-1))      # comparisons with NaN are False, so the ends are never peaks
print(w[peaks].index.tolist())

[1, 3, 6]


## You're done with the learning workbook

Next: open **`pandas_exam.ipynb`**. Rules for the exam:
- Predict the Part A answers *before* running anything.
- No peeking at this workbook for Parts B to D on your first pass. If you get stuck, move on and note the question number.
- Run the last cell (`check(globals())`) for your score, then paste your answers or tell me your score and I'll walk through every mistake with you.